In [28]:
!pip install -q groq langsmith
import os
from google.colab import userdata
from groq import Groq
from groq.resources.chat.completions import Completions
from langsmith import traceable

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("langsmith_api_key")
os.environ["LANGSMITH_PROJECT"] = "agentic-ai-session1"

# Log every Groq call in this notebook to LangSmith
keep = ("model", "messages", "temperature", "reasoning_effort", "response_format", "max_tokens")
Completions.create = traceable(name="Groq chat", run_type="llm",
                               process_inputs=lambda i: {k: v for k, v in i.items() if k in keep})(Completions.create)

client = Groq(api_key=userdata.get("groq_api_key"))
MODEL = "openai/gpt-oss-120b"

In [29]:
from google.colab import userdata
from groq import Groq
from collections import Counter

client = Groq(api_key=userdata.get("groq_api_key"))
MODEL = "openai/gpt-oss-120b"

def ask_many_times(prompt, runs=15):
    answers = []
    for _ in range(runs):
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=1.0,          # same temperature every time
            max_tokens=2000,          # plenty of room for thinking + answer
            reasoning_effort="low",   # think briefly
            include_reasoning=False,  # return only the final answer
        )
        answer = (response.choices[0].message.content or "").strip().lower().rstrip(".!")
        answers.append(answer or "(empty)")
    return Counter(answers)

def show(title, counts):
    print(f"\n{title}  ->  {len(counts)} different answers")
    for answer, count in counts.most_common():
        print(f"  {'█' * count} {count}  {answer}")

In [30]:
!pip install -q langchain langchain-groq langchain-text-splitters langchain-huggingface sentence-transformers

import os
from google.colab import userdata
from IPython.display import Markdown

In [31]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = userdata.get("langsmith_api_key")
os.environ["LANGSMITH_PROJECT"] = "shopeasy-rag1"
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [32]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b")

In [33]:
%%writefile shopeasy_handbook.txt
# ShopEasy – Customer Support Handbook

**Document Type:** Customer Support Knowledge Base
**Version:** 1.0

---

## 1. Return Policy

ShopEasy allows customers to return eligible products within 10 calendar days of delivery.

Products must be unused, undamaged and returned with their original packaging, tags and accessories.

The following products are not eligible for return:
- Personal care and hygiene products.
- Perishable food items.
- Customized products.
- Products damaged due to customer misuse.

Customers must raise a return request through their ShopEasy account.

Once the return request is approved, a pickup will be scheduled within 2–4 business days.

## 2. Refund Policy

Refunds are initiated after the returned product passes the quality inspection.

The refund processing time depends on the payment method:

- Credit Card: 5–7 business days.
- Debit Card: 5–7 business days.
- UPI: 2–4 business days.
- Net Banking: 5–7 business days.
- ShopEasy Wallet: Within 24 hours.

Refunds are generally credited to the original payment method.

If a refund is not received within the specified period, customers should contact customer support with their order ID and refund reference number.

## 3. Shipping and Delivery Policy

ShopEasy delivers products across most serviceable locations in India.

Standard delivery generally takes 3–7 business days after order confirmation.

Express delivery is available in selected locations and generally takes 1–2 business days.

Customers can track their orders through the My Orders section of their ShopEasy account.

Shipping charges:
- Orders above ₹999: Free standard delivery.
- Orders below ₹999: ₹49 standard delivery charge.
- Express delivery: Additional charges may apply.

Delivery timelines may vary due to weather conditions, public holidays or operational delays.

## 4. Order Cancellation Policy

Customers can cancel an order before it is shipped.

Once an order has been shipped, cancellation through the website is not available.

Customers may request a return after delivery, subject to the return policy.

For prepaid orders cancelled before shipment, refunds are initiated to the original payment method.

For Cash on Delivery orders, no refund is required if cancellation occurs before shipment.

## 5. Payment Methods

ShopEasy supports the following payment methods:

- UPI payments.
- Credit cards.
- Debit cards.
- Net banking.
- ShopEasy Wallet.
- Cash on Delivery (COD).

Cash on Delivery is available only for eligible products and serviceable locations.

Customers should never share their OTP, PIN, CVV or banking passwords with anyone claiming to represent ShopEasy.

## 6. Exchange Policy

Customers can request an exchange within 10 calendar days of product delivery.

Exchanges are subject to product availability and eligibility.

The replacement product must be of the same product category and should meet the return eligibility conditions.

If the requested replacement product is unavailable, customers may request a refund according to the refund policy.

## 7. Customer Support

ShopEasy customer support is available Monday to Saturday, from 9:00 AM to 6:00 PM.

Customers can contact support through:

- Email: support@shopeasy.example
- Customer Support Number: 1800-000-0000
- Website: My Account → Help and Support

Customers should provide their order ID when raising a complaint or requesting assistance.

---

## 8. Frequently Asked Questions

**Q1. Can I return a product after 15 days?**

No. The standard return window is 10 calendar days from delivery.

**Q2. How long does a UPI refund take?**

UPI refunds generally take 2–4 business days after the refund is initiated.

**Q3. Can I cancel an order after shipping?**

No. Once an order is shipped, cancellation through the website is unavailable.

**Q4. Is standard delivery free?**

Yes, standard delivery is free for orders above ₹999.

**Q5. Can I return a used product?**

No. Products must be unused and meet the return eligibility requirements.

**Q6. Do you provide international delivery?**

This handbook does not specify international delivery availability. Customers should contact customer support for confirmation.

**Q7. Can I exchange a product if the replacement is unavailable?**

Yes. Customers may request a refund according to the refund policy.

---

**End of Document**

*Note: This handbook is fictional practice data created for learning RAG, document chunking, embeddings, vector databases and retrieval-based question answering.*

Overwriting shopeasy_handbook.txt


In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
handbook=open('shopeasy_handbook.txt').read()
questions={'Where is my order ORD101?','What is status of refund for RORD101',
'How long does a refund take?','What payment methods are supported?',
'Can I cancel an order after shipping?',
'Do you provide international delivery?','What is your exchange policy?'}
splitter_chunks=RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter_chunks.create_documents([handbook])
print(len(chunks), "chunks")
for chunk in chunks[:10]:
    print(chunk.page_content, "\n-----")

18 chunks
# ShopEasy – Customer Support Handbook

**Document Type:** Customer Support Knowledge Base
**Version:** 1.0

---

## 1. Return Policy

ShopEasy allows customers to return eligible products within 10 calendar days of delivery. 
-----
Products must be unused, undamaged and returned with their original packaging, tags and accessories.

The following products are not eligible for return:
- Personal care and hygiene products.
- Perishable food items.
- Customized products.
- Products damaged due to customer misuse. 
-----
Customers must raise a return request through their ShopEasy account.

Once the return request is approved, a pickup will be scheduled within 2–4 business days.

## 2. Refund Policy

Refunds are initiated after the returned product passes the quality inspection. 
-----
The refund processing time depends on the payment method:

- Credit Card: 5–7 business days.
- Debit Card: 5–7 business days.
- UPI: 2–4 business days.
- Net Banking: 5–7 business days.
- ShopEasy 

In [35]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
from langchain_core.vectorstores import InMemoryVectorStore

# InMemoryVectorStore, a lightweight vector store that keeps all embeddings and documents in the system memory (RAM).

vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)

In [37]:
from langchain.tools import tool
from langchain.agents import create_agent
@tool
def get_order_status(order_id: str):
   'Track the status for orders based on order_id'
   orders = {
        "ORD101": "Shipped",
        "ORD102": "Delivered",
        "ORD103": "Processing"
    }
   return orders.get(order_id, "Order not found")
@tool
def get_refund_status(order_id: str):
    'Track the refund status for orders based on refund_id'
    refunds = {
        "RORD101": "Intiated",
        "RORD102": "Done",
        "RORD103": "Processing"
    }
    return refunds.get(order_id, "Refund not found")
@tool
def search_handbook(query: str) -> str:
    """Search ShopEasy's customer policy handbook for returns, refunds, delivery, payments and contact details."""
    results = vector_store.similarity_search(query, k=3)
    return "\n\n".join(doc.page_content for doc in results)
def ask_agent(system,user):
  agent = create_agent(
    model=llm,
    tools=[search_handbook,get_order_status,get_refund_status],
    system_prompt=system,
)
  response = agent.invoke({
        "messages": [
            {"role": "user", "content": user}
        ]
    })

  return response

In [38]:
for q in questions:
    results = vector_store.similarity_search_with_score(q, k=3)
    result=ask_agent("You are ShopEasy's customer support assistant. If you can't find the answer, say you don't know. give ans in not more than 2 to 3 lines.",q)
    for message in result["messages"]:
       message.pretty_print()

================================ Human Message =================================

What is status of refund for RORD101
================================== Ai Message ==================================
Tool Calls:
  get_refund_status (fc_af863cac-bd27-4ad7-9161-49bd66561822)
 Call ID: fc_af863cac-bd27-4ad7-9161-49bd66561822
  Args:
    order_id: RORD101
================================= Tool Message =================================
Name: get_refund_status

Intiated
================================== Ai Message ==================================

The refund for order **RORD101** is currently **Initiated**. If you need further assistance, feel free to let us know.
================================ Human Message =================================

Do you provide international delivery?
================================== Ai Message ==================================
Tool Calls:
  search_handbook (fc_ac20fc80-aec0-4f0f-858a-43e3c73c87f3)
 Call ID: fc_ac20fc80-aec0-4f0f-858a-43e3c73c87f3
  Args

In [39]:
from typing import Literal
from pydantic import BaseModel, ConfigDict
class insurance_class(BaseModel):
  model_config = ConfigDict(extra="forbid")
  customer_name:str
  policy_number:str
  event_date:str
  event_location:str
  event_description:str
  claim_amount:float
  supporting_documents:list[str]
  claim_type:Literal["Health", "Vehicle", "Home", "Other"]

In [40]:
def ask_insurance(system,user):
  response=client.chat.completions.create(
      model=MODEL,
      messages=[{'role':'system','content':system},{'role':'user','content':user}],
      reasoning_effort='low',
      response_format={                     # 👈 the only new part
            "type": "json_schema",
            "json_schema": {"name": "ticket1", "schema": insurance_class.model_json_schema()},
        },
  )
  result =insurance_class.model_validate_json(response.choices[0].message.content)
  return result

In [41]:
emails=['''Hello Insurance Team,

My name is Rahul Sharma. My policy number is HOME-2026-1005.

On 2 November 2026, a water pipe burst in my kitchen
at my home in Chandigarh. This damaged my kitchen
cabinets and flooring.

The estimated repair cost is Rs. 65,000.
I have attached photographs and a plumber's quotation.

Please help me file a claim.

Regards,
Rahul''','''Subject: Request for Home Insurance Claim – Water Damage

Dear ABC Insurance Team,

My name is Rahul Sharma and my policy number is HOME-2026-1005.

On 2 November 2026, a water pipe suddenly burst in my kitchen
at my residence in Chandigarh. The incident caused damage to
my kitchen cabinets, wooden flooring and nearby electrical fittings.

The estimated repair cost is Rs. 65,000.

I have attached photographs of the damaged area and a
plumber's repair quotation.

Please register my claim and let me know the next steps.

Regards,
Rahul Sharma''','''Subject: Home Insurance Claim – Water Leakage Damage

Dear ABC Insurance Team,

My name is Priya Mehta and my policy number is HOME-2026-1006.

For the last three months, water has been slowly leaking
from a pipe behind my bedroom wall. Recently, I noticed
damage to the wall paint, wooden furniture and flooring.

The estimated repair cost is Rs. 42,000.

I have attached photographs of the damaged property and
a contractor's repair estimate.

Please help me submit an insurance claim.

Regards,
Priya Mehta''','''Subject: Theft Claim – Missing Household Items

Dear ABC Insurance Team,

My name is Amit Verma and my policy number is HOME-2026-1007.

On 4 November 2026, I noticed that my laptop, smartwatch
and some jewellery were missing from my house in Ludhiana.

There were no signs of forced entry. The doors and windows
were properly locked when I left the house.

The estimated value of the missing items is Rs. 1,20,000.

I have attached photographs of the missing items and
the purchase invoices. I have also registered a police
complaint and attached a copy.

Please process my insurance claim.

Regards,
Amit Verma''','''Subject: Home Insurance Claim – Electrical Fire Damage

Dear ABC Insurance Team,

My name is Neha Kapoor and my policy number is HOME-2026-1008.

On 5 November 2026, an electrical short circuit occurred
in my living room, resulting in an accidental fire.

The fire damaged my television, sofa, curtains and
part of the living room ceiling.

The estimated repair and replacement cost is Rs. 85,000.

I have attached photographs of the damaged property,
the electrician's inspection report and repair estimates.

Kindly register my insurance claim.

Regards,
Neha Kapoor''','''Subject: Home Insurance Claim – Earthquake Damage

Dear ABC Insurance Team,

My name is Karan Malhotra and my policy number is.

On 6 November 2026, an earthquake caused cracks in the
walls and ceiling of my residential property in Amritsar.

Some furniture and household items were also damaged.

The estimated repair cost is Rs. 2,50,000.

I have attached photographs of the damaged property
and a structural engineer's inspection report.

Please let me know whether this damage is covered
under my home insurance policy.

Regards,
Karan Malhotra''']

In [42]:
def check_data(result):
  if result.policy_number is None or (isinstance(result.policy_number, str) and result.policy_number.strip().upper() in ["", "N/A", "NA", "NONE"]):
      data_check='Manual Review'
  elif result.claim_amount<=0:
      data_check='Invalid Claim Amount'
  elif not result.supporting_documents:
      print('Documents Required')
      data_check='Documents Required'
  else:
      data_check='Ready for Processing'
  return data_check

In [43]:
class insurance_status(BaseModel):
  model_config=ConfigDict(extra='forbid')
  coverage_status:Literal['Covered','Not Covered','Not Specified']
  explanation:str
  suporting_policy_revidence:str

In [63]:
def search_the_policy(result):
  description=result.event_description
  claim_amount=result.claim_amount
  question = f"""
    Does the policy cover this {claim_amount} insurance claim?
    Incident description: {description}
    """
  results = vector_store1.similarity_search_with_score(question, k=3)

  context = "\n\n".join(
        doc.page_content for doc, score in results
    )
  system=''' system = """
You are an insurance claim policy evaluation assistant.

Evaluate the claim using ONLY the retrieved policy context.

You must always return a valid JSON response
matching the provided schema.

Never respond with plain text such as
"I do not know."

Classification rules:

1. Covered:
   The policy explicitly covers the incident
   and all applicable conditions are satisfied.

2. Not Covered:
   The policy explicitly excludes the incident
   or a required coverage condition is not satisfied.

3. Not Specified:
   The policy does not mention the incident.

4. Manual Review:
   Retrieved evidence is insufficient or ambiguous.

If the answer is unknown, return:
coverage_status = "Manual Review"

Do not invent policy conditions or use general insurance knowledge.

Always populate all three required fields.
If policy evidence is unavailable, use an empty string.
"""
    '''

  response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": system + "\n\nPolicy Context:\n" + context
            },
            {
                "role": "user",
                "content": question
            }
        ],
        reasoning_effort='low',
        response_format={                     # 👈 the only new part
            "type": "json_schema",
            "json_schema": {"name": "ticket1", "schema": insurance_status.model_json_schema()},
        },
  )

  return insurance_status.model_validate_json(response.choices[0].message.content)

In [45]:
question = "Does the insurance policy cover accidental water damage caused by a sudden pipe burst?"

results = vector_store1.similarity_search_with_score(question, k=3)

for doc, score in results:
    print("SCORE:", score)
    print("CONTENT:", doc.page_content)
    print("-" * 50)

SCORE: 0.773273410019676
CONTENT: # ABC Insurance – Home Insurance Policy

## Section 1: Water Damage Coverage
The policy covers accidental water damage caused by sudden pipe bursts, plumbing failures, and unexpected water leakage inside the insured property.
--------------------------------------------------
SCORE: 0.582430807711399
CONTENT: Damage caused by gradual leakage, poor maintenance, or negligence is not covered.

## Section 2: Fire Damage Coverage
Fire damage caused by accidental fire, electrical short circuits, or unexpected incidents is covered.

Damage caused intentionally by the policyholder is excluded.
--------------------------------------------------
SCORE: 0.4709702516026112
CONTENT: ## Section 5: Claim Submission Period
Customers must report an incident within 7 calendar days of discovering the damage.

Late reporting may require additional review by the insurance company.
--------------------------------------------------


In [46]:
def check_policy(result,status):
  if status=='Ready for Processing':
    return search_the_policy(result)


In [64]:
for e in emails:
  system='''you are working in insurance company.you will receive customer claim emails.Your task is to  read the email, extracts important details and represent the output
  in required format.If you do not know the ans then mention i do not know and do not invent new data and false information,'''
  result=ask_insurance(system,e)
  result1=check_data(result)
  print(f'Customer:{result.customer_name}\nPolicy:{result.policy_number}\nAmount:{result.claim_amount}\nEvent_description:{result.event_description}\nStatus:{result1}\nclaim_type:{result.claim_type}')
  print(' '*60)
  if result1=='Ready for Processing':
    result2=check_policy(result,result1)
    validation_check='PASS'
    if result2.coverage_status=='Covered':
      coverage_status='Covered'
    elif result2.coverage_status=='Not Covered':
      coverage_status='Not Covered'
    else:
      coverage_status='Manual Review'
    print(f'''Validation:{validation_check}\nCoverage_status:{result2.coverage_status}\nexplanation:{result2.explanation}\nsuporting_policy_revidence:{result2.suporting_policy_revidence}\nFinal Decision:{coverage_status}''')
  else:
    validation_check='Failed'
    print(f'Validation:{validation_check}\nResult:Not Executed\nFinal Decision:Manual Review')
  print(' '*60)

Customer:Rahul Sharma
Policy:HOME-2026-1005
Amount:65000.0
Event_description:Water pipe burst in kitchen causing damage to cabinets and flooring.
Status:Ready for Processing
claim_type:Home
                                                            
Validation:PASS
Coverage_status:Covered
explanation:The incident describes a water pipe burst, which is listed as an accidental water damage cause covered by the policy. The damage to cabinets and flooring falls under the covered water damage.
suporting_policy_revidence:Section 1: Water Damage Coverage – "The policy covers accidental water damage caused by sudden pipe bursts, plumbing failures, and unexpected water leakage inside the insured property."
Final Decision:Covered
                                                            
Customer:Rahul Sharma
Policy:HOME-2026-1005
Amount:65000.0
Event_description:Water pipe burst in kitchen causing damage to kitchen cabinets, wooden flooring, and nearby electrical fittings.
Status:Ready for P

In [48]:
%%writefile insurance_policy.txt
# ABC Insurance – Home Insurance Policy

## Section 1: Water Damage Coverage
The policy covers accidental water damage caused by sudden pipe bursts, plumbing failures, and unexpected water leakage inside the insured property.

Damage caused by gradual leakage, poor maintenance, or negligence is not covered.

## Section 2: Fire Damage Coverage
Fire damage caused by accidental fire, electrical short circuits, or unexpected incidents is covered.

Damage caused intentionally by the policyholder is excluded.

## Section 3: Theft Coverage
The policy covers theft of household belongings when there is evidence of forced entry and a police complaint has been registered.

Theft without evidence of forced entry is not covered.

## Section 4: Claim Documentation
Customers must submit:
- A completed claim application.
- Photographs or videos of the damaged property.
- Repair estimates or invoices.
- A police report for theft-related claims.

The insurance company may request additional documents when required.

## Section 5: Claim Submission Period
Customers must report an incident within 7 calendar days of discovering the damage.

Late reporting may require additional review by the insurance company.

## Section 6: Exclusions
The following are not covered:
- Normal wear and tear.
- Intentional damage.
- Damage caused by poor maintenance.
- Damage caused by gradual deterioration.
- Losses resulting from negligence.

Overwriting insurance_policy.txt


In [49]:
handbook=open('insurance_policy.txt').read()
chunks_splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=chunks_splitter.create_documents([handbook])
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
from langchain_core.vectorstores import InMemoryVectorStore

# InMemoryVectorStore, a lightweight vector store that keeps all embeddings and documents in the system memory (RAM).

vector_store1 = InMemoryVectorStore.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [50]:

def ask_policy(system, user):
    results = vector_store1.similarity_search_with_score(user, k=3)

    context = "\n\n".join(
        doc.page_content for doc, score in results
    )

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                'role': 'system',
                'content': system + '\n\nContext:\n' + context
            },
            {
                'role': 'user',
                'content': user
            }
        ],
        reasoning_effort='low',
    )

    return response.choices[0].message.content


In [51]:
questions=['Does insurance cover accidental water pipe bursts?',
'Is gradual water leakage covered?',
'Is theft without forced entry covered?',
'How many days do I have to report damage?',
'Does the policy cover earthquake damage?']

In [52]:
for e in questions:
  system=f'''you are working in  insurance company that  receives hundreds of claims.Before processing a claim,you need to check whether the incident is covered under its insurance policy.Answer only using the context below. If the answer is not there, say you don't know.'''
  output=ask_policy(system,e)
  print(output)

Yes, accidental water pipe bursts are covered under the policy.
Gradual water leakage is not covered.
No, theft without evidence of forced entry is not covered.
You have 7 calendar days to report the damage.
I don't know.
